# R Master v8a Fix2 · Head Frame Alignment

Fix1 已证实：头部 donor 成功提取，但坐标系没有被正确固化，导致 **Lap 头与 Mona 身体分离 / 飞头**。

Fix2 改为：
- Lapine donor 在独立 Blender 进程中 **烘焙为统一世界坐标**；
- 以 **Head + LeftEye + RightEye** 建立完整头部坐标框架；
- Mona 使用 **Head + Eye.L + Eye.R** 建立目标框架；
- 用眼距决定统一缩放，用完整三轴框架决定旋转；
- 以 Head 骨根作为接头锚点；
- 加入硬 sanity audit：位置、眼距、头/身尺度异常直接停止，不再生成“飞头”图；
- Mona 原头不再删顶点，而用末端 Mask 隐藏，避免破坏 CorrectiveSmooth 顶点数。

依旧是 preview：
- 505 骨不改；
- 不转权重；
- 不焊 neck；
- 不烘焙 Rest Pose；
- 不导 VRM。


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, zipfile, tarfile, json, os, hashlib

BUILD_TAG="v8a_fix2_headframe_20260920_r1"
print("R Master v8a Fix2 · Head Frame Alignment")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v7c_refine"/"latest"/"R_Master_v7c_REFINED_PREVIEW.blend"
REF=ROOT/"reference"
CACHE=ROOT/"cache"
CP=ROOT/"v8a_head_graft"/"checkpoint"
OUT=ROOT/"v8a_head_graft"/"fix2_latest"
for p in (REF,CACHE,CP,OUT): p.mkdir(parents=True,exist_ok=True)

LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if not SRC.exists(): raise RuntimeError("v7c source missing")
if not LAPZIP.exists(): raise RuntimeError("Lapine Drive cache missing")
print(f"✓ v7c {SRC.stat().st_size/1024/1024:.1f} MiB")
print(f"✓ Lapine {LAPZIP.stat().st_size/1024/1024:.1f} MiB")

h=hashlib.sha256()
with LAPZIP.open("rb") as f:
    for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
sha=h.hexdigest()
EXPECTED="8e215aeae8d2e42719305d66292b84c2714f616b7952591dee0383c376c86af4"
print("Lapine SHA256:",sha)
if sha!=EXPECTED: raise RuntimeError("Lapine source hash mismatch")
print("✓ source audit pass")


In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v8a_fix2"); LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender ready")

UNPACK=LOCAL/"lapine_unpack"
if UNPACK.exists(): shutil.rmtree(UNPACK)
UNPACK.mkdir()
outer=UNPACK/"outer"
with zipfile.ZipFile(LAPZIP,"r") as z:
    unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
    if not unity: raise RuntimeError("unitypackage missing")
    z.extract(unity[0],outer)
UNITY=outer/unity[0]
pkg=UNPACK/"pkg"; pkg.mkdir()
with tarfile.open(UNITY,"r:*") as t: t.extractall(pkg)
FBX=None
for p in pkg.glob("*/pathname"):
    try:q=p.read_text(encoding="utf-8").strip()
    except:continue
    if q=="Assets/Models/FBX/Lapine.fbx":
        a=p.parent/"asset"
        if a.exists():
            FBX=UNPACK/"Lapine.fbx"; shutil.copy2(a,FBX); break
if FBX is None: raise RuntimeError("Lapine.fbx missing")
print(f"✓ Lapine.fbx {FBX.stat().st_size/1024/1024:.1f} MiB")


In [ ]:
DONOR_BLEND=CP/"Lapine_Head_Donor_v8a_fix2.blend"
DONOR_META=CP/"Lapine_Head_Donor_v8a_fix2.json"
DONOR_SCRIPT=LOCAL/"extract_donor_fix2.py"
DONOR_SCRIPT.write_text("\nimport bpy,sys,os,json\nfrom mathutils import Vector,Matrix\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nfbx=out=meta=None\nfor i,a in enumerate(argv):\n    if a==\"--fbx\": fbx=argv[i+1]\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--meta\": meta=argv[i+1]\nif not all([fbx,out,meta]): raise RuntimeError(\"args\")\n\nbpy.ops.wm.read_factory_settings(use_empty=True)\nbpy.ops.import_scene.fbx(filepath=fbx,automatic_bone_orientation=False)\nrig=max([o for o in bpy.data.objects if o.type==\"ARMATURE\"],key=lambda o:len(o.data.bones))\nrig.data.pose_position=\"REST\"\nbpy.context.view_layer.update()\n\ndef dbone(names):\n    for n in names:\n        b=rig.data.bones.get(n)\n        if b:return b\n    for b in rig.data.bones:\n        if any(b.name.lower().endswith(n.lower()) for n in names): return b\n    return None\n\nhead=dbone([\"Head\"])\nle=dbone([\"LeftEye\"])\nre=dbone([\"RightEye\"])\nif not all([head,le,re]): raise RuntimeError(\"Lap Head/eyes anchors missing\")\n\ndef wh(b): return rig.matrix_world@b.head_local\ndef wt(b): return rig.matrix_world@b.tail_local\nH,L,R=wh(head),wh(le),wh(re)\nUP=(wt(head)-H).normalized()\nRIGHT=(R-L).normalized()\nFWD=RIGHT.cross(UP).normalized()\nRIGHT=UP.cross(FWD).normalized()\neye_dist=(R-L).length\nif eye_dist<1e-6: raise RuntimeError(\"Lap eye spacing invalid\")\n\ntokens=(\"face\",\"eye\",\"brow\",\"hair\",\"lash\",\"ear\")\nsrc=[o for o in bpy.data.objects if o.type==\"MESH\" and any(t in o.name.lower() for t in tokens)]\nif len(src)<3: raise RuntimeError(\"too few Lap head meshes\")\n\ndeps=bpy.context.evaluated_depsgraph_get()\nnew=[]\nfor o in src:\n    eo=o.evaluated_get(deps)\n    me=bpy.data.meshes.new_from_object(eo,depsgraph=deps)\n    # bake evaluated mesh into the same WORLD coordinate system as H/L/R\n    M=o.matrix_world.copy()\n    for v in me.vertices:\n        v.co=M@v.co\n    no=bpy.data.objects.new(\"LapDonor_\"+o.name,me)\n    bpy.context.collection.objects.link(no)\n    no.matrix_world=Matrix.Identity(4)\n    no[\"R_DONOR\"]=\"LapineHead_v8a_fix2\"\n    new.append(no)\n\nfor o in list(bpy.data.objects):\n    if o not in new:\n        bpy.data.objects.remove(o,do_unlink=True)\n\ndef bounds(objs):\n    pts=[]\n    for o in objs:\n        pts += [o.matrix_world@Vector(c) for c in o.bound_box]\n    mn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\n    mx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\n    return mn,mx\n\nmn,mx=bounds(new)\ninfo={\n \"head_origin\":list(map(float,H)),\n \"left_eye\":list(map(float,L)),\n \"right_eye\":list(map(float,R)),\n \"up\":list(map(float,UP)),\n \"right\":list(map(float,RIGHT)),\n \"forward\":list(map(float,FWD)),\n \"eye_distance\":float(eye_dist),\n \"bbox_min\":list(map(float,mn)),\n \"bbox_max\":list(map(float,mx)),\n \"meshes\":[o.name for o in new],\n \"mesh_count\":len(new),\n \"world_baked\":True\n}\nbpy.ops.wm.save_as_mainfile(filepath=out,check_existing=False)\njson.dump(info,open(meta,\"w\",encoding=\"utf-8\"),ensure_ascii=False,indent=2)\nprint(\"[v8a Fix2] DONOR_OK\",len(new),eye_dist,H)\n",encoding="utf-8")
need=True
if DONOR_BLEND.exists() and DONOR_BLEND.stat().st_size>2*1024*1024 and DONOR_META.exists():
    try:
        m=json.loads(DONOR_META.read_text(encoding="utf-8"))
        need=not m.get("world_baked") or m.get("mesh_count",0)<3
    except: need=True
if need:
    print("① Bake Lap head into one world coordinate frame")
    log=CP/"donor_fix2.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background","--factory-startup","--python",str(DONOR_SCRIPT),
         "--","--fbx",str(FBX),"--out",str(DONOR_BLEND),"--meta",str(DONOR_META)]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8a Fix2]" in line or "Traceback" in line or "Error" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("donor bake failed")
else: print("✓ Fix2 donor checkpoint exists")
print(json.loads(DONOR_META.read_text(encoding="utf-8")))


In [ ]:
GRAFT_SCRIPT=LOCAL/"graft_fix2.py"
GRAFT_SCRIPT.write_text("\nimport bpy,sys,os,json,math\nfrom mathutils import Vector,Matrix\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\ndonor=meta=out=tag=None\nfor i,a in enumerate(argv):\n    if a==\"--donor\": donor=argv[i+1]\n    if a==\"--meta\": meta=argv[i+1]\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--tag\": tag=argv[i+1]\nif not all([donor,meta,out]): raise RuntimeError(\"args\")\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or not rig: raise RuntimeError(\"Mona source missing\")\nrig.data.pose_position=\"REST\"\nbpy.context.view_layer.update()\n\ndef dbone(names):\n    for n in names:\n        b=rig.data.bones.get(n)\n        if b:return b\n    for b in rig.data.bones:\n        if any(b.name.lower().endswith(n.lower()) for n in names): return b\n    return None\nhead=dbone([\"Head\"])\nle=dbone([\"Eye.L\"])\nre=dbone([\"Eye.R\"])\nif not all([head,le,re]): raise RuntimeError(\"Mona Head/Eye anchors missing\")\ndef wh(b): return rig.matrix_world@b.head_local\ndef wt(b): return rig.matrix_world@b.tail_local\nTH,TL,TR=wh(head),wh(le),wh(re)\nTUP=(wt(head)-TH).normalized()\nTRIGHT=(TR-TL).normalized()\nTFWD=TRIGHT.cross(TUP).normalized()\nTRIGHT=TUP.cross(TFWD).normalized()\nteye=(TR-TL).length\nif teye<1e-6: raise RuntimeError(\"Mona eye spacing invalid\")\n\ndm=json.load(open(meta,\"r\",encoding=\"utf-8\"))\nDH=Vector(dm[\"head_origin\"]); DL=Vector(dm[\"left_eye\"]); DR=Vector(dm[\"right_eye\"])\nDUP=Vector(dm[\"up\"]); DRIGHT=Vector(dm[\"right\"]); DFWD=Vector(dm[\"forward\"])\ndeye=float(dm[\"eye_distance\"])\nscale=teye/deye\n\n# basis columns: Right, Forward, Up\nDB=Matrix(((DRIGHT.x,DFWD.x,DUP.x),(DRIGHT.y,DFWD.y,DUP.y),(DRIGHT.z,DFWD.z,DUP.z)))\nTB=Matrix(((TRIGHT.x,TFWD.x,TUP.x),(TRIGHT.y,TFWD.y,TUP.y),(TRIGHT.z,TFWD.z,TUP.z)))\nROT=TB@DB.transposed()\nX=Matrix.Translation(TH)@ROT.to_4x4()@Matrix.Scale(scale,4)@Matrix.Translation(-DH)\n\nbefore=set(bpy.data.objects)\nwith bpy.data.libraries.load(donor,link=False) as (src,dst):\n    dst.objects=list(src.objects)\nfor o in dst.objects:\n    if o is not None:bpy.context.collection.objects.link(o)\ndonors=[o for o in bpy.data.objects if o not in before and o.type==\"MESH\"]\nif not donors: raise RuntimeError(\"0 donor meshes\")\nfor o in donors:\n    o.matrix_world=X\n    o[\"R_DONOR\"]=\"LapineHead_v8a_fix2\"\n\n# Hard alignment audit\nhead_err=(X@DH-TH).length\neye_dist_after=(X@DR-X@DL).length\neye_ratio_err=abs(eye_dist_after-teye)/teye\n\n# Hide Mona head with a final MASK modifier; do not delete topology.\nfor m in list(body.modifiers):\n    if m.name==\"R_v8a_Fix2_Hide_Mona_Head\": body.modifiers.remove(m)\nif \"R_v8a_Fix2_HeadHide\" in body.vertex_groups:\n    vg=body.vertex_groups[\"R_v8a_Fix2_HeadHide\"]\n    vg.remove(range(len(body.data.vertices)))\nelse:\n    vg=body.vertex_groups.new(name=\"R_v8a_Fix2_HeadHide\")\n\ngids=[body.vertex_groups[n].index for n in (\"Head\",\"HeadStretch\") if n in body.vertex_groups]\nhide=[]\nmw=body.matrix_world.copy()\nhead_base_z=TH.z\nfor v in body.data.vertices:\n    p=mw@v.co\n    weights={g.group:g.weight for g in v.groups}\n    hw=max([weights.get(i,0.0) for i in gids] or [0.0])\n    if p.z>head_base_z-0.015 and hw>0.05:\n        hide.append(v.index)\nif len(hide)<100: raise RuntimeError(\"Mona head mask too small\")\nvg.add(hide,1.0,\"REPLACE\")\nmask=body.modifiers.new(\"R_v8a_Fix2_Hide_Mona_Head\",\"MASK\")\nmask.vertex_group=vg.name\nmask.invert_vertex_group=True\n\n# Remove other mesh objects from body shell preview.\nfor o in list(bpy.data.objects):\n    if o.type==\"MESH\" and o!=body and o not in donors:\n        bpy.data.objects.remove(o,do_unlink=True)\n\ndef bbox(objs):\n    pts=[]\n    for o in objs: pts += [o.matrix_world@Vector(c) for c in o.bound_box]\n    mn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\n    mx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\n    return mn,mx\ndmn,dmx=bbox(donors)\nbmn,bmx=bbox([body])\nbody_h=bmx.z-bmn.z\ndonor_h=dmx.z-dmn.z\nhead_center=(dmn+dmx)*.5\ncenter_dist=(head_center-TH).length\nheight_ratio=donor_h/body_h\n\naudit={\n \"head_anchor_error_m\":float(head_err),\n \"target_eye_distance_m\":float(teye),\n \"donor_eye_distance_after_m\":float(eye_dist_after),\n \"eye_distance_relative_error\":float(eye_ratio_err),\n \"donor_head_height_m\":float(donor_h),\n \"body_height_m\":float(body_h),\n \"head_to_body_height_ratio\":float(height_ratio),\n \"donor_bbox_center_to_head_anchor_m\":float(center_dist)\n}\nif head_err>1e-5: raise RuntimeError(\"head anchor audit fail \"+str(audit))\nif eye_ratio_err>1e-4: raise RuntimeError(\"eye scale audit fail \"+str(audit))\nif not (0.08 < height_ratio < 0.40): raise RuntimeError(\"head/body size audit fail \"+str(audit))\nif center_dist>0.50: raise RuntimeError(\"head position audit fail \"+str(audit))\n\nrep={\"ok\":True,\"stage\":\"R_Master_v8a_Fix2_HeadFrameAlignment\",\"build_tag\":tag,\n     \"mona_bone_count\":len(rig.data.bones),\"donor_mesh_count\":len(donors),\n     \"donor_meshes\":[o.name for o in donors],\"mona_head_mask_vertex_count\":len(hide),\n     \"uniform_scale_from_eye_distance\":float(scale),\"alignment_audit\":audit,\n     \"preview_only\":True,\"weights_transferred\":False,\"neck_welded\":False,\n     \"rest_pose_baked\":False,\"final_vrm\":False}\nos.makedirs(out,exist_ok=True)\njson.dump(rep,open(os.path.join(out,\"R_Master_v8a_Fix2_report.json\"),\"w\",encoding=\"utf-8\"),ensure_ascii=False,indent=2)\nbpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,\"R_Master_v8a_Fix2_HEAD_GRAFT_PREVIEW.blend\"),check_existing=False)\nprint(\"[v8a Fix2] GRAFT_OK\",json.dumps(audit))\n",encoding="utf-8")
STAGE=OUT/"R_Master_v8a_Fix2_HEAD_GRAFT_PREVIEW.blend"
REPORT=OUT/"R_Master_v8a_Fix2_report.json"
need=True
if STAGE.exists() and STAGE.stat().st_size>50*1024*1024 and REPORT.exists():
    try: need=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except: need=True
if need:
    print("② Build head graft with 3-axis head frame + hard audit")
    TMP=LOCAL/"graft"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir()
    log=TMP/"graft_fix2.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(GRAFT_SCRIPT),
         "--","--donor",str(DONOR_BLEND),"--meta",str(DONOR_META),"--out",str(TMP),"--tag",BUILD_TAG]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8a Fix2]" in line or "Traceback" in line or "RuntimeError" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("Fix2 graft stopped by build/audit")
    for n in ("R_Master_v8a_Fix2_HEAD_GRAFT_PREVIEW.blend","R_Master_v8a_Fix2_report.json"):
        shutil.copy2(TMP/n,OUT/n)
    shutil.copy2(log,OUT/"R_Master_v8a_Fix2_graft.log")
else: print("✓ Fix2 graft checkpoint exists")
print(json.dumps(json.loads(REPORT.read_text(encoding="utf-8")),ensure_ascii=False,indent=2))


In [ ]:
RENDER_SCRIPT=LOCAL/"render_fix2.py"
RENDER_SCRIPT.write_text("\nimport bpy,os,sys\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--view\": view=argv[i+1]\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix2\"]\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or not donors or not rig: raise RuntimeError(\"preview objects missing\")\n\nfor o in bpy.context.scene.objects:\n    if o.type==\"ARMATURE\": o.hide_render=True\n    if o.type==\"MESH\": o.hide_render=(o!=body and o not in donors)\n\n# Frame full body from Mona body only so a donor anomaly cannot shrink Mona to a dot.\nbp=[body.matrix_world@Vector(c) for c in body.bound_box]\nbmn=Vector((min(p.x for p in bp),min(p.y for p in bp),min(p.z for p in bp)))\nbmx=Vector((max(p.x for p in bp),max(p.y for p in bp),max(p.z for p in bp)))\nbc=(bmn+bmx)*.5; h=bmx.z-bmn.z; d=h*2.6\n\nhb=rig.data.bones.get(\"Head\")\nHC=rig.matrix_world@hb.head_local if hb else Vector((bc.x,bc.y,bmx.z-h*.12))\n\ns=bpy.context.scene\ns.render.engine=\"BLENDER_WORKBENCH\"; s.render.image_settings.file_format=\"PNG\"\ns.display.shading.light=\"STUDIO\"; s.display.shading.show_shadows=True; s.display.shading.show_cavity=True\ns.display.shading.cavity_type=\"WORLD\"; s.display.shading.color_type=\"SINGLE\"; s.display.shading.single_color=(.62,.62,.65)\ns.display.shading.background_type=\"VIEWPORT\"; s.display.shading.background_color=(.04,.04,.05)\ncd=bpy.data.cameras.new(\"R_v8a_fix2_cam_data\"); cam=bpy.data.objects.new(\"R_v8a_fix2_cam\",cd); s.collection.objects.link(cam); s.camera=cam\ncam.data.type=\"ORTHO\"\ndef look(t): cam.rotation_euler=(Vector(t)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\ndef rr(fn,pos,tgt,scale,res):\n    s.render.resolution_x,s.render.resolution_y=res; s.render.resolution_percentage=100\n    cam.location=Vector(pos); cam.data.ortho_scale=scale; look(tgt); s.render.filepath=os.path.join(out,fn); bpy.ops.render.render(write_still=True)\n\nspec={\n\"front\":(\"R_Master_v8a_Fix2_front.png\",(bc.x,bc.y-d,bc.z),bc,h*1.08,(640,900)),\n\"side\":(\"R_Master_v8a_Fix2_side.png\",(bc.x+d,bc.y,bc.z),bc,h*1.08,(640,900)),\n\"back\":(\"R_Master_v8a_Fix2_back.png\",(bc.x,bc.y+d,bc.z),bc,h*1.08,(640,900)),\n\"three_quarter\":(\"R_Master_v8a_Fix2_three_quarter.png\",(bc.x+d*.72,bc.y-d*.72,bc.z),bc,h*1.08,(640,900)),\n\"head_front\":(\"R_Master_v8a_Fix2_head_front.png\",(HC.x,HC.y-d,HC.z+h*.08),HC+Vector((0,0,h*.08)),h*.30,(800,800)),\n\"head_side\":(\"R_Master_v8a_Fix2_head_side.png\",(HC.x+d,HC.y,HC.z+h*.08),HC+Vector((0,0,h*.08)),h*.30,(800,800)),\n\"head_three_quarter\":(\"R_Master_v8a_Fix2_head_three_quarter.png\",(HC.x+d*.72,HC.y-d*.72,HC.z+h*.08),HC+Vector((0,0,h*.08)),h*.30,(800,800)),\n\"neck_interface\":(\"R_Master_v8a_Fix2_neck_interface.png\",(HC.x+d*.70,HC.y-d*.70,HC.z-h*.01),HC+Vector((0,0,-h*.01)),h*.23,(800,650))}\nrr(*spec[view])\nprint(\"[v8a Fix2] RENDER_OK\",view)\n",encoding="utf-8")
views=[("front","R_Master_v8a_Fix2_front.png"),("side","R_Master_v8a_Fix2_side.png"),
("back","R_Master_v8a_Fix2_back.png"),("three_quarter","R_Master_v8a_Fix2_three_quarter.png"),
("head_front","R_Master_v8a_Fix2_head_front.png"),("head_side","R_Master_v8a_Fix2_head_side.png"),
("head_three_quarter","R_Master_v8a_Fix2_head_three_quarter.png"),("neck_interface","R_Master_v8a_Fix2_neck_interface.png")]
for i,(v,f) in enumerate(views,1):
    p=OUT/f
    if p.exists() and p.stat().st_size>15000:
        print(f"✓ [{i}/8] {v} checkpoint"); continue
    print(f"③ [{i}/8] {v}")
    r=subprocess.run(["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER_SCRIPT),
                      "--","--out",str(OUT),"--view",v],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if r.returncode!=0:
        print(r.stdout[-10000:]); raise RuntimeError(v+" render failed")
print("✓ renders complete")


In [ ]:
from google.colab import files
from IPython.display import display,Image,Markdown
items=[("全身正面","R_Master_v8a_Fix2_front.png"),("全身侧面","R_Master_v8a_Fix2_side.png"),
("全身背面","R_Master_v8a_Fix2_back.png"),("全身 3/4","R_Master_v8a_Fix2_three_quarter.png"),
("头肩正面","R_Master_v8a_Fix2_head_front.png"),("头肩侧面","R_Master_v8a_Fix2_head_side.png"),
("头肩 3/4","R_Master_v8a_Fix2_head_three_quarter.png"),("颈部接口","R_Master_v8a_Fix2_neck_interface.png")]
for title,f in items:
    display(Markdown("### "+title)); display(Image(filename=str(OUT/f),width=480))
z=OUT/"R_Master_v8a_Fix2_Review.zip"
if z.exists(): z.unlink()
with zipfile.ZipFile(z,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as w:
    for _,f in items:w.write(OUT/f,arcname=f)
    for f in ("R_Master_v8a_Fix2_report.json","R_Master_v8a_Fix2_graft.log"):
        if (OUT/f).exists():w.write(OUT/f,arcname=f)
print(f"✓ Review ZIP {z.stat().st_size/1024/1024:.1f} MiB")
files.download(str(z))
